
# Robust Regression Under Outliers and Heavy-Tailed Noise

## Objective
This project compares three regression methods under simulated data scenarios where standard linear regression assumptions may be violated:

1. **Ordinary Least Squares (OLS)**
2. **Huber Regression**
3. **Quantile Regression**

The project uses **three simulated scenarios**:

- **Scenario 1: Clean Data**  
- **Scenario 2: Heavy-Tailed Noise**  
- **Scenario 3: Outlier Contamination**

The goal is to examine how robust methods behave when data contain heavy-tailed noise or outliers.


## 1. Import Libraries

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, HuberRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import statsmodels.api as sm

np.random.seed(42)



## 2. Data-Generating Process

We simulate data from a simple linear model:

\[
y = 3 + 2X + \varepsilon
\]

where:
- **X** is the predictor
- **y** is the response
- **\varepsilon** is the noise term

We generate three datasets:
1. **Clean data** with Gaussian noise
2. **Heavy-tailed data** with Student-t noise
3. **Outlier-contaminated data** where some response values are shifted strongly upward


In [ ]:

def generate_clean_data(n=200):
    X = np.random.uniform(0, 10, n)
    noise = np.random.normal(0, 2, n)
    y = 3 + 2 * X + noise
    return X.reshape(-1, 1), y

def generate_heavy_tailed_data(n=200):
    X = np.random.uniform(0, 10, n)
    noise = np.random.standard_t(df=2, size=n) * 2
    y = 3 + 2 * X + noise
    return X.reshape(-1, 1), y

def generate_outlier_data(n=200, outlier_fraction=0.10):
    X = np.random.uniform(0, 10, n)
    noise = np.random.normal(0, 2, n)
    y = 3 + 2 * X + noise

    num_outliers = int(n * outlier_fraction)
    outlier_indices = np.random.choice(n, num_outliers, replace=False)
    y[outlier_indices] = y[outlier_indices] + np.random.normal(20, 5, num_outliers)

    return X.reshape(-1, 1), y


## 3. Generate the Simulated Datasets

In [ ]:

X_clean, y_clean = generate_clean_data()
X_heavy, y_heavy = generate_heavy_tailed_data()
X_outlier, y_outlier = generate_outlier_data()


## 4. Visualize the Three Scenarios

In [ ]:

def plot_dataset(X, y, title):
    plt.figure(figsize=(7, 4.5))
    plt.scatter(X, y, alpha=0.7)
    plt.title(title)
    plt.xlabel("X")
    plt.ylabel("y")
    plt.show()

plot_dataset(X_clean, y_clean, "Scenario 1: Clean Data")
plot_dataset(X_heavy, y_heavy, "Scenario 2: Heavy-Tailed Noise")
plot_dataset(X_outlier, y_outlier, "Scenario 3: Outlier Contamination")



## 5. Define Regression Methods

We fit three models:

### A. Ordinary Least Squares (OLS)
Standard linear regression that minimizes squared errors.

### B. Huber Regression
A robust regression method that reduces the influence of outliers.

### C. Quantile Regression
A regression method that estimates conditional quantiles. Here we use the **median** (0.5 quantile), which is often more robust than mean-based regression.


In [ ]:

def fit_ols(X, y):
    model = LinearRegression()
    model.fit(X, y)
    predictions = model.predict(X)
    return model, predictions

def fit_huber(X, y):
    model = HuberRegressor()
    model.fit(X, y)
    predictions = model.predict(X)
    return model, predictions

def fit_quantile(X, y, q=0.5):
    X_q = sm.add_constant(X)
    model = sm.QuantReg(y, X_q).fit(q=q)
    predictions = model.predict(X_q)
    return model, predictions


## 6. Evaluation Metrics

In [ ]:

def evaluate_model(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    return rmse, mae


## 7. Compare Models on a Single Scenario

In [ ]:

# Example: Clean data
ols_model_clean, ols_pred_clean = fit_ols(X_clean, y_clean)
huber_model_clean, huber_pred_clean = fit_huber(X_clean, y_clean)
quant_model_clean, quant_pred_clean = fit_quantile(X_clean, y_clean)

print("Clean Data Performance")
print("OLS RMSE, MAE:", evaluate_model(y_clean, ols_pred_clean))
print("Huber RMSE, MAE:", evaluate_model(y_clean, huber_pred_clean))
print("Quantile RMSE, MAE:", evaluate_model(y_clean, quant_pred_clean))


## 8. Create a Function to Compare All Models

In [ ]:

def compare_models(X, y, scenario_name):
    results = []

    # OLS
    ols_model, ols_pred = fit_ols(X, y)
    ols_rmse, ols_mae = evaluate_model(y, ols_pred)
    results.append([scenario_name, "OLS", ols_rmse, ols_mae])

    # Huber
    huber_model, huber_pred = fit_huber(X, y)
    huber_rmse, huber_mae = evaluate_model(y, huber_pred)
    results.append([scenario_name, "Huber", huber_rmse, huber_mae])

    # Quantile
    quant_model, quant_pred = fit_quantile(X, y)
    quant_rmse, quant_mae = evaluate_model(y, quant_pred)
    results.append([scenario_name, "Quantile", quant_rmse, quant_mae])

    return results


## 9. Run Comparison on All Three Scenarios

In [ ]:

results = []
results.extend(compare_models(X_clean, y_clean, "Clean"))
results.extend(compare_models(X_heavy, y_heavy, "Heavy-Tailed"))
results.extend(compare_models(X_outlier, y_outlier, "Outliers"))

results_df = pd.DataFrame(results, columns=["Scenario", "Method", "RMSE", "MAE"])
results_df


## 10. Sort Results

In [ ]:
results_df.sort_values(by=['Scenario', 'RMSE'])

## 11. Plot RMSE Comparison

In [ ]:

for scenario in results_df["Scenario"].unique():
    subset = results_df[results_df["Scenario"] == scenario]

    plt.figure(figsize=(6, 4))
    plt.bar(subset["Method"], subset["RMSE"])
    plt.title(f"RMSE Comparison - {scenario}")
    plt.xlabel("Method")
    plt.ylabel("RMSE")
    plt.show()


## 12. Plot MAE Comparison

In [ ]:

for scenario in results_df["Scenario"].unique():
    subset = results_df[results_df["Scenario"] == scenario]

    plt.figure(figsize=(6, 4))
    plt.bar(subset["Method"], subset["MAE"])
    plt.title(f"MAE Comparison - {scenario}")
    plt.xlabel("Method")
    plt.ylabel("MAE")
    plt.show()


## 13. Plot Fitted Regression Lines on the Outlier Scenario

In [ ]:

ols_model, ols_pred = fit_ols(X_outlier, y_outlier)
huber_model, huber_pred = fit_huber(X_outlier, y_outlier)
quant_model, quant_pred = fit_quantile(X_outlier, y_outlier)

sorted_idx = np.argsort(X_outlier[:, 0])
X_sorted = X_outlier[sorted_idx]
ols_pred_sorted = ols_pred[sorted_idx]
huber_pred_sorted = huber_pred[sorted_idx]
quant_pred_sorted = quant_pred[sorted_idx]

plt.figure(figsize=(8, 5))
plt.scatter(X_outlier, y_outlier, alpha=0.6, label="Data")
plt.plot(X_sorted, ols_pred_sorted, label="OLS")
plt.plot(X_sorted, huber_pred_sorted, label="Huber")
plt.plot(X_sorted, quant_pred_sorted, label="Quantile")
plt.title("Regression Fits on Outlier Data")
plt.xlabel("X")
plt.ylabel("y")
plt.legend()
plt.show()



## 14. Coefficient Comparison

The true data-generating relationship is:

\[
y = 3 + 2X + \varepsilon
\]

So the true intercept is **3** and the true slope is **2**.

Below we compare estimated coefficients for each method under the outlier scenario.


In [ ]:

# OLS coefficients
ols_intercept = ols_model.intercept_
ols_slope = ols_model.coef_[0]

# Huber coefficients
huber_intercept = huber_model.intercept_
huber_slope = huber_model.coef_[0]

# Quantile coefficients
quant_intercept = quant_model.params[0]
quant_slope = quant_model.params[1]

coef_df = pd.DataFrame({
    "Method": ["OLS", "Huber", "Quantile"],
    "Intercept": [ols_intercept, huber_intercept, quant_intercept],
    "Slope": [ols_slope, huber_slope, quant_slope]
})

coef_df



## 15. Interpretation of Results

When you review the output, focus on the following:

### Clean Data
- OLS often performs very well because its assumptions are approximately correct.
- Robust methods may perform similarly or slightly worse when no contamination is present.

### Heavy-Tailed Noise
- Heavy-tailed noise creates more extreme values than Gaussian noise.
- Robust methods such as Huber or Quantile regression may become more competitive.

### Outlier Contamination
- OLS is sensitive to large outliers because it minimizes squared errors.
- Huber and Quantile regression usually reduce the influence of those extreme points and can produce more stable fits.

This is the main statistical message of the project:  
**model performance depends on whether the assumptions behind the model are satisfied.**



## 16. Final Conclusion

This project compared **OLS**, **Huber regression**, and **Quantile regression** under three simulated settings:

1. Clean linear data
2. Heavy-tailed noise
3. Outlier contamination

### Main takeaway
- **OLS** is often strong when data are clean.
- **Robust methods** become more useful when data contain outliers or heavy-tailed noise.
- This demonstrates why robust statistical methods matter when regression assumptions are violated.

### Why this project matters
The project is a simple simulation-based study of robustness in regression. It is relevant to statistics and machine learning because real-world data often contain contamination, outliers, or departures from ideal assumptions.
